# Analisi linee murine (D2A1, D2.0R) — unità = cellula

Ogni **sottocartella** della cartella di una giornata è una **cellula**.
- **Forza-indentazione**: ogni curva della cellula (di solito 3) è fittata con Ting; la cellula è rappresentata dalla media delle sue curve.
- **Stress-relaxation**: le curve della cellula sono mediate in una curva della cellula, che viene poi fittata (SLS, PLR, bi-exp, poroelastico).

Alla fine: tabella per cellula, curva SR media di ciascuna linea, test di Mann–Whitney tra le due linee (una osservazione per cellula).

**Da modificare:** solo la cella 2 (cartelle delle due linee).

**Nota:** gli output già salvati precedono la rampa di chiusura al contatto e i nuovi criteri di esclusione. Rieseguire la cella di analisi completa per aggiornare i risultati in `aggregated/per_cellula_murine_chiusura_contatto`.

In [ ]:
# 1. Import
import sys
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo')
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/general/src')

import importlib
import warnings
import matplotlib.pyplot as plt
import analisi_per_cellula as apc
importlib.reload(apc)

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

In [ ]:
# 2. Configurazione
SISTEMA = "murine"

# Per ogni linea: le cartelle delle giornate. Dentro ciascuna: <cellula>/FD e <cellula>/SR
LINEE = {
    "D2A1": [
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/140326/A1",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/240326/A1",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/270326/A1",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/100426/A1/SAMPLE1",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/100426/A1/SAMPLE2",
        # aggiungi qui le altre giornate D2A1
    ],
    "D2.0R": [
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/160426/0R",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/140326/0R",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/240326/0R",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/150426/0R/SAMPLE1",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/150426/0R/SAMPLE2",
        # aggiungi qui le altre giornate D2.0R
    ],
}

OUTPUT_DIR = r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/per_cellula_murine_chiusura_contatto"
ESEGUI_FD = True
ESEGUI_SR = True
OPZIONI_TING = {"param_overrides": {"apply_ramp_correction": True}}

# Criteri di esclusione (gli stessi per le due linee). Per non escludere nulla: CRITERI = None
# fd_r2_min: R2 minimo del fit di Ting; fd_max_rapporto_hertz_E0: E_Hertz/E0 tra 1/3 e 3;
# sr_F_relax_fisico: 0 < F_relax <= 1; sr_r2_min: R2 minimo del miglior modello della fase tardiva
CRITERI = dict(apc.CRITERI_MURINE)
CRITERI["sr_F_relax_fisico"] = False
ALTEZZE_UM = {"D2A1": 4.6, "D2.0R": 4.6}

In [ ]:
# 3. Controllo: quali cellule vengono trovate (nessun fit)
cells = apc.find_cells(LINEE, SISTEMA)
apc.print_cells_overview(cells)
display(cells[["linea", "giornata", "cartella", "tipo", "id_cellula", "n_curve"]].head(20))

In [ ]:
# 4. Analisi completa: FD (Ting) e SR, tabelle per cellula, medie per linea, test
risultati = apc.run_full_analysis(
    LINEE,
    SISTEMA,
    OUTPUT_DIR,
    run_fd=ESEGUI_FD,
    run_sr=ESEGUI_SR,
    criteri=CRITERI,
    ting_options=OPZIONI_TING,
    altezze_um=ALTEZZE_UM,
)

In [ ]:
# 5. Indentazione durante il mantenimento (curve di stress-relaxation)
# Calcola delta(t) durante la pausa per ogni curva SR e la sua variazione relativa massima.
import os
import pandas as pd
import sr_utils as sr
importlib.reload(sr)

cells_sr = apc.find_cells(LINEE, SISTEMA)
cells_sr = cells_sr[cells_sr["tipo"] == "SR"]
if globals().get("ESCLUDI_GIORNATE"):
    cells_sr = cells_sr[~cells_sr["giornata"].astype(str).str.contains("|".join(ESCLUDI_GIORNATE))]

tabelle = []
ris_tesi = {}
for linea, gruppo in cells_sr.groupby("linea", sort=False):
    files = [f for lista in gruppo["curve"] for f in lista]
    nome = apc.NOMI_BREVI_TESI.get(linea, linea)
    print(f"--- {nome}")
    tab, ris = sr.variazione_indentazione_pausa(files)
    tab.insert(0, "linea", nome)
    tabelle.append(tab)
    ris_tesi[nome] = ris
    sr.grafico_indentazione_pausa(ris, titolo=nome,
                                  path_pdf=os.path.join(OUTPUT_DIR, f"delta_pausa_{SISTEMA}_{nome}.pdf"))
    plt.show()

sr.figura_tesi_indentazione_pausa(ris_tesi, os.path.join(OUTPUT_DIR, f"fig_tesi_delta_pausa_{SISTEMA}.pdf"))
plt.show()

tab_delta = pd.concat(tabelle, ignore_index=True)
tab_delta.to_csv(os.path.join(OUTPUT_DIR, f"delta_pausa_{SISTEMA}.csv"), index=False)
display(tab_delta[tab_delta["valida"]].groupby("linea")[["delta_0_um", "variazione_max_nm", "variazione_relativa_max_pct"]]
        .agg(["mean", "std", "median", "max"]).round(2))